In [ ]:
import pandas as pd
import numpy as sns
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import os

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


listings = pd.read_csv('/content/drive/MyDrive/AirBNB analytics/listings.csv')
display(listings.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,id,listing_url,scrape_id,last_scraped,name,summary,space,description,experiences_offered,neighborhood_overview,...,review_scores_value,requires_license,license,jurisdiction_names,instant_bookable,cancellation_policy,require_guest_profile_picture,require_guest_phone_verification,calculated_host_listings_count,reviews_per_month
0,241032,https://www.airbnb.com/rooms/241032,20160104002432,2016-01-04,Stylish Queen Anne Apartment,NaN,Make your self at home in this charming one-be...,Make your self at home in this charming one-be...,none,NaN,...,10.0,f,NaN,WASHINGTON,f,moderate,f,f,2,4.07
1,953595,https://www.airbnb.com/rooms/953595,20160104002432,2016-01-04,Bright & Airy Queen Anne Apartment,Chemically sensitive? We've removed the irrita...,"Beautiful, hypoallergenic apartment in an extr...",Chemically sensitive? We've removed the irrita...,none,"Queen Anne is a wonderful, truly functional vi...",...,10.0,f,NaN,WASHINGTON,f,strict,t,t,6,1.48
2,3308979,https://www.airbnb.com/rooms/3308979,20160104002432,2016-01-04,New Modern House-Amazing water view,New modern house built in 2013. Spectacular s...,"Our house is modern, light and fresh with a wa...",New modern house built in 2013. Spectacular s...,none,Upper Queen Anne is a charming neighborhood fu...,...,10.0,f,NaN,WASHINGTON,f,strict,f,f,2,1.15
3,7421966,https://www.airbnb.com/rooms/7421966,20160104002432,2016-01-04,Queen Anne Chateau,A charming apartment that sits atop Queen Anne...,NaN,A charming apartment that sits atop Queen Anne...,none,NaN,...,NaN,f,NaN,WASHINGTON,f,flexible,f,f,1,NaN
4,278830,https://www.airbnb.com/rooms/278830,20160104002432,2016-01-04,Charming craftsman 3 bdm house,Cozy family craftman house in beautiful neighb...,Cozy family craftman house in beautiful neighb...,Cozy family craftman house in beautiful neighb...,none,We are in the beautiful neighborhood of Queen ...,...,9.0,f,NaN,WASHINGTON,f,strict,f,f,1,0.89


Outlier detection function

In [ ]:
def detect_outliers(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    return outliers

listings data cleaning

In [ ]:
listings.head()

,id,listing_url,scrape_id,last_scraped,name,summary,space,description,experiences_offered,neighborhood_overview,...,review_scores_value,requires_license,license,jurisdiction_names,instant_bookable,cancellation_policy,require_guest_profile_picture,require_guest_phone_verification,calculated_host_listings_count,reviews_per_month
0,241032,https://www.airbnb.com/rooms/241032,20160104002432,2016-01-04,Stylish Queen Anne Apartment,NaN,Make your self at home in this charming one-be...,Make your self at home in this charming one-be...,none,NaN,...,10.0,f,NaN,WASHINGTON,f,moderate,f,f,2,4.07
1,953595,https://www.airbnb.com/rooms/953595,20160104002432,2016-01-04,Bright & Airy Queen Anne Apartment,Chemically sensitive? We've removed the irrita...,"Beautiful, hypoallergenic apartment in an extr...",Chemically sensitive? We've removed the irrita...,none,"Queen Anne is a wonderful, truly functional vi...",...,10.0,f,NaN,WASHINGTON,f,strict,t,t,6,1.48
2,3308979,https://www.airbnb.com/rooms/3308979,20160104002432,2016-01-04,New Modern House-Amazing water view,New modern house built in 2013. Spectacular s...,"Our house is modern, light and fresh with a wa...",New modern house built in 2013. Spectacular s...,none,Upper Queen Anne is a charming neighborhood fu...,...,10.0,f,NaN,WASHINGTON,f,strict,f,f,2,1.15
3,7421966,https://www.airbnb.com/rooms/7421966,20160104002432,2016-01-04,Queen Anne Chateau,A charming apartment that sits atop Queen Anne...,NaN,A charming apartment that sits atop Queen Anne...,none,NaN,...,NaN,f,NaN,WASHINGTON,f,flexible,f,f,1,NaN
4,278830,https://www.airbnb.com/rooms/278830,20160104002432,2016-01-04,Charming craftsman 3 bdm house,Cozy family craftman house in beautiful neighb...,Cozy family craftman house in beautiful neighb...,Cozy family craftman house in beautiful neighb...,none,We are in the beautiful neighborhood of Queen ...,...,9.0,f,NaN,WASHINGTON,f,strict,f,f,1,0.89


In [ ]:
listings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3818 entries, 0 to 3817
Data columns (total 92 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   id                                3818 non-null   int64  
 1   listing_url                       3818 non-null   object 
 2   scrape_id                         3818 non-null   int64  
 3   last_scraped                      3818 non-null   object 
 4   name                              3818 non-null   object 
 5   summary                           3641 non-null   object 
 6   space                             3249 non-null   object 
 7   description                       3818 non-null   object 
 8   experiences_offered               3818 non-null   object 
 9   neighborhood_overview             2786 non-null   object 
 10  notes                             2212 non-null   object 
 11  transit                           2884 non-null   object 
 12  thumbn

In [ ]:
listings.shape

(3818, 92)

Checking for duplicates on the basis of id column

In [ ]:
print(f"Number of duplicate listings based on 'id': {listings.duplicated(subset=['id']).sum()}")
print("Duplicate listings (if any):")
listings[listings.duplicated(subset=['id'], keep=False)]

Number of duplicate listings based on 'id': 0
Duplicate listings (if any):


,id,listing_url,scrape_id,last_scraped,name,summary,space,description,experiences_offered,neighborhood_overview,...,review_scores_value,requires_license,license,jurisdiction_names,instant_bookable,cancellation_policy,require_guest_profile_picture,require_guest_phone_verification,calculated_host_listings_count,reviews_per_month


In [ ]:
listings.isnull().sum()

,0
id,0
listing_url,0
scrape_id,0
last_scraped,0
name,0
...,...
cancellation_policy,0
require_guest_profile_picture,0
require_guest_phone_verification,0
calculated_host_listings_count,0


Before cleaning data only selecting the rows needed for analysis

In [ ]:
listings = listings[['id', 'name', 'experiences_offered',
                     'transit', 'host_id', 'host_name', 'host_since', 'host_location',
                     'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_is_superhost', 'host_neighbourhood',
                     'host_listings_count', 'host_total_listings_count', 'host_verifications', 'neighbourhood', 'neighbourhood_cleansed',
                     'neighbourhood_group_cleansed', 'city', 'state', 'zipcode', 'market', 'smart_location', 'country_code',
                     'latitude', 'longitude', 'is_location_exact', 'property_type', 'room_type', 'accommodates', 'bathrooms', 'bedrooms',
                     'beds', 'bed_type', 'amenities', 'square_feet', 'price', 'weekly_price', 'monthly_price', 'security_deposit', 'cleaning_fee',
                     'guests_included', 'extra_people', 'minimum_nights', 'maximum_nights', 'calendar_updated', 'has_availability', 'availability_30',
                     'availability_60', 'availability_90', 'availability_365', 'number_of_reviews',
                     'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_communication',
                     'review_scores_location', 'review_scores_value',
                     'instant_bookable', 'cancellation_policy', 'calculated_host_listings_count', 'reviews_per_month']]

In [ ]:
listings.shape

(3818, 64)

In [ ]:
listings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3818 entries, 0 to 3817
Data columns (total 64 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              3818 non-null   int64  
 1   name                            3818 non-null   object 
 2   experiences_offered             3818 non-null   object 
 3   transit                         2884 non-null   object 
 4   host_id                         3818 non-null   int64  
 5   host_name                       3816 non-null   object 
 6   host_since                      3816 non-null   object 
 7   host_location                   3810 non-null   object 
 8   host_response_time              3295 non-null   object 
 9   host_response_rate              3295 non-null   object 
 10  host_acceptance_rate            3045 non-null   object 
 11  host_is_superhost               3816 non-null   object 
 12  host_neighbourhood              35

Columns like price, weekly_price, monthly_price, security_deposit, cleaning_fee, and extra_people contain string formats ($1,250.00). Strip $ and ,, then cast to float64.

In [ ]:
columns_to_clean = ['price', 'weekly_price', 'monthly_price', 'security_deposit', 'cleaning_fee', 'extra_people']

for col in columns_to_clean:
    if col in listings.columns:
        # Ensure the column is treated as string before string operations
        # Fill NaN with an empty string temporarily to avoid errors with .str.replace(), then convert to float.
        # This ensures NaN values remain NaN after conversion.
        listings[col] = listings[col].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(float)
        print(f"Cleaned and converted '{col}' to float64.")
    else:
        print(f"Column '{col}' not found in the DataFrame.")

# Verify the changes
print("\nData types after cleaning:")
display(listings[columns_to_clean].info())

Cleaned and converted 'price' to float64.
Cleaned and converted 'weekly_price' to float64.
Cleaned and converted 'monthly_price' to float64.
Cleaned and converted 'security_deposit' to float64.
Cleaned and converted 'cleaning_fee' to float64.
Cleaned and converted 'extra_people' to float64.

Data types after cleaning:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3818 entries, 0 to 3817
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   price             3818 non-null   float64
 1   weekly_price      2009 non-null   float64
 2   monthly_price     1517 non-null   float64
 3   security_deposit  1866 non-null   float64
 4   cleaning_fee      2788 non-null   float64
 5   extra_people      3818 non-null   float64
dtypes: float64(6)
memory usage: 179.1 KB


None

Parse Percentages: host_response_rate and host_acceptance_rate contain %. Strip % and divide by 100 to convert to float (0.95).

In [ ]:
percentage_columns = ['host_response_rate', 'host_acceptance_rate']

for col in percentage_columns:
    if col in listings.columns:
        # Convert to string to handle string operations, replace NaN with empty string for .str.replace
        # Then convert to float after cleaning, ensuring NaN values are preserved
        listings[col] = listings[col].astype(str).str.replace('%', '', regex=False).astype(float) / 100
        print(f"Cleaned and converted '{col}' to float (0-1 range).")
    else:
        print(f"Column '{col}' not found in the DataFrame.")

# Verify the changes
print("\nData types and first few rows after cleaning percentages:")
display(listings[percentage_columns].info())
display(listings[percentage_columns].head())

Cleaned and converted 'host_response_rate' to float (0-1 range).
Cleaned and converted 'host_acceptance_rate' to float (0-1 range).

Data types and first few rows after cleaning percentages:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3818 entries, 0 to 3817
Data columns (total 2 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   host_response_rate    3295 non-null   float64
 1   host_acceptance_rate  3045 non-null   float64
dtypes: float64(2)
memory usage: 59.8 KB


None

,host_response_rate,host_acceptance_rate
0,0.96,1.0
1,0.98,1.0
2,0.67,1.0
3,NaN,NaN
4,1.00,NaN


Boolean Conversions: Convert host_is_superhost, is_location_exact, has_availability, instant_bookable, require_guest_profile_picture, and require_guest_phone_verification from 't'/'f' strings to boolean (True/False).

In [ ]:
boolean_columns = [
    'host_is_superhost',
    'is_location_exact',
    'has_availability',
    'instant_bookable'
]

for col in boolean_columns:
    if col in listings.columns:
        # Map 't' to True, 'f' to False. Handle NaN values by letting them remain NaN.
        listings[col] = listings[col].map({'t': True, 'f': False}).astype(bool)
        print(f"Converted '{col}' to boolean.")
    else:
        print(f"Column '{col}' not found in the DataFrame.")

# Verify the changes
print("\nData types and first few rows after boolean conversion:")
display(listings[boolean_columns].info())
display(listings[boolean_columns].head())

Converted 'host_is_superhost' to boolean.
Converted 'is_location_exact' to boolean.
Converted 'has_availability' to boolean.
Converted 'instant_bookable' to boolean.

Data types and first few rows after boolean conversion:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3818 entries, 0 to 3817
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   host_is_superhost  3818 non-null   bool 
 1   is_location_exact  3818 non-null   bool 
 2   has_availability   3818 non-null   bool 
 3   instant_bookable   3818 non-null   bool 
dtypes: bool(4)
memory usage: 15.0 KB


None

,host_is_superhost,is_location_exact,has_availability,instant_bookable
0,False,True,True,False
1,True,True,True,False
2,False,True,True,False
3,False,True,True,False
4,False,True,True,False


High Missingness Dropping: Drop columns with $>80\%$ missing data that offer low signal.

In [ ]:
missing_percentages = listings.isnull().sum() / len(listings) * 100
high_missingness_columns = missing_percentages[missing_percentages >= 80].index.tolist()

print(f"Columns with more than 80% missing values: {len(high_missingness_columns)} columns")
if high_missingness_columns:
    for col in high_missingness_columns:
        print(f"- {col}: {missing_percentages[col]:.2f}% missing")
else:
    print("No columns found with more than 80% missing values.")

# Code to drop these columns (uncomment and run if desired)
listings = listings.drop(columns=high_missingness_columns)
print(f"\nDropped {len(high_missingness_columns)} columns with high missingness.")
print("New listings shape:", listings.shape)

Columns with more than 80% missing values: 1 columns
- square_feet: 97.46% missing

Dropped 1 columns with high missingness.
New listings shape: (3818, 63)


In [ ]:
listings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3818 entries, 0 to 3817
Data columns (total 63 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              3818 non-null   int64  
 1   name                            3818 non-null   object 
 2   experiences_offered             3818 non-null   object 
 3   transit                         2884 non-null   object 
 4   host_id                         3818 non-null   int64  
 5   host_name                       3816 non-null   object 
 6   host_since                      3816 non-null   object 
 7   host_location                   3810 non-null   object 
 8   host_response_time              3295 non-null   object 
 9   host_response_rate              3295 non-null   float64
 10  host_acceptance_rate            3045 non-null   float64
 11  host_is_superhost               3818 non-null   bool   
 12  host_neighbourhood              35

In [ ]:
listings['host_response_time'].value_counts(dropna=False)

,count
host_response_time,
within an hour,1692
within a few hours,968
within a day,597
NaN,523
a few days or more,38


In [ ]:
for col in listings.select_dtypes(include='object').columns:
    if col == 'zipcode':
        print(f"Skipping imputation for '{col}' as requested.")
        continue
    if listings[col].isnull().any():
        listings[col] = listings[col].fillna('unknown')
        print(f"Imputed null values in '{col}' with 'unknown'.")

Imputed null values in 'transit' with 'unknown'.
Imputed null values in 'host_name' with 'unknown'.
Imputed null values in 'host_since' with 'unknown'.
Imputed null values in 'host_location' with 'unknown'.
Imputed null values in 'host_response_time' with 'unknown'.
Imputed null values in 'host_neighbourhood' with 'unknown'.
Imputed null values in 'host_verifications' with 'unknown'.
Imputed null values in 'neighbourhood' with 'unknown'.
Skipping imputation for 'zipcode' as requested.
Imputed null values in 'property_type' with 'unknown'.


zipcode, market, state $\rightarrow$ Mode imputation within neighbourhood_cleansed.

In [ ]:
columns_for_mode_imputation = ['zipcode', 'market', 'state']

for col in columns_for_mode_imputation:
    if col in listings.columns:
        # Fill NaN values with the mode within each 'neighbourhood_cleansed' group
        listings[col] = listings.groupby('neighbourhood_cleansed')[col].transform(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 'unknown'))
        print(f"Imputed null values in '{col}' with mode within 'neighbourhood_cleansed' groups.")
    else:
        print(f"Column '{col}' not found in the DataFrame.")

print("\nVerifying null counts after mode imputation:")
display(listings[columns_for_mode_imputation].isnull().sum())

Imputed null values in 'zipcode' with mode within 'neighbourhood_cleansed' groups.
Imputed null values in 'market' with mode within 'neighbourhood_cleansed' groups.
Imputed null values in 'state' with mode within 'neighbourhood_cleansed' groups.

Verifying null counts after mode imputation:


,0
zipcode,0
market,0
state,0


bathrooms, bedrooms, beds $\rightarrow$ Median imputation conditioned on room_type and accommodates.

In [ ]:
columns_for_median_imputation = ['bathrooms', 'bedrooms', 'beds']

for col in columns_for_median_imputation:
    if col in listings.columns:
        listings[col] = listings.groupby(['room_type', 'accommodates'])[col].transform(lambda x: x.fillna(x.median()))
        print(f"Imputed null values in '{col}' with median within 'room_type' and 'accommodates' groups.")
    else:
        print(f"Column '{col}' not found in the DataFrame.")

print("\nVerifying null counts after median imputation:")
display(listings[columns_for_median_imputation].isnull().sum())

Imputed null values in 'bathrooms' with median within 'room_type' and 'accommodates' groups.
Imputed null values in 'bedrooms' with median within 'room_type' and 'accommodates' groups.
Imputed null values in 'beds' with median within 'room_type' and 'accommodates' groups.

Verifying null counts after median imputation:


,0
bathrooms,0
bedrooms,0
beds,0


security_deposit, cleaning_fee $\rightarrow$ Impute 0.0 (missing typically means no fee charged).

In [ ]:
columns_to_impute_zero = ['security_deposit', 'cleaning_fee']

for col in columns_to_impute_zero:
    if col in listings.columns:
        listings[col] = listings[col].fillna(0.0)
        print(f"Imputed null values in '{col}' with 0.0.")
    else:
        print(f"Column '{col}' not found in the DataFrame.")

print("\nVerifying null counts after 0.0 imputation:")
display(listings[columns_to_impute_zero].isnull().sum())

Imputed null values in 'security_deposit' with 0.0.
Imputed null values in 'cleaning_fee' with 0.0.

Verifying null counts after 0.0 imputation:


,0
security_deposit,0
cleaning_fee,0


review_scores_* columns $\rightarrow$ Create an indicator flag has_reviews (1 if present, 0 if null), then impute missing ratings with the column median

In [ ]:
review_score_columns = [col for col in listings.columns if 'review_scores_' in col]

# 1. Create has_reviews indicator flag
if 'review_scores_rating' in listings.columns:
    listings['has_reviews'] = listings['review_scores_rating'].notna().astype(int)
    print("Created 'has_reviews' indicator (1 if review_scores_rating is present, 0 if null).")
else:
    print("Column 'review_scores_rating' not found, 'has_reviews' not created.")

# 2. Impute missing ratings with the column median
for col in review_score_columns:
    if col in listings.columns:
        if listings[col].isnull().any():
            median_value = listings[col].median()
            listings[col] = listings[col].fillna(median_value)
            print(f"Imputed null values in '{col}' with median: {median_value}")
    else:
        print(f"Column '{col}' not found in the DataFrame.")

print("\nVerifying null counts after review scores imputation:")
display(listings[['has_reviews'] + review_score_columns].isnull().sum())

Created 'has_reviews' indicator (1 if review_scores_rating is present, 0 if null).
Imputed null values in 'review_scores_rating' with median: 96.0
Imputed null values in 'review_scores_accuracy' with median: 10.0
Imputed null values in 'review_scores_cleanliness' with median: 10.0
Imputed null values in 'review_scores_checkin' with median: 10.0
Imputed null values in 'review_scores_communication' with median: 10.0
Imputed null values in 'review_scores_location' with median: 10.0
Imputed null values in 'review_scores_value' with median: 10.0

Verifying null counts after review scores imputation:


,0
has_reviews,0
review_scores_rating,0
review_scores_accuracy,0
review_scores_cleanliness,0
review_scores_checkin,0
review_scores_communication,0
review_scores_location,0
review_scores_value,0


host_acceptance_rate

In [ ]:
listings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3818 entries, 0 to 3817
Data columns (total 64 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              3818 non-null   int64  
 1   name                            3818 non-null   object 
 2   experiences_offered             3818 non-null   object 
 3   transit                         3818 non-null   object 
 4   host_id                         3818 non-null   int64  
 5   host_name                       3818 non-null   object 
 6   host_since                      3818 non-null   object 
 7   host_location                   3818 non-null   object 
 8   host_response_time              3818 non-null   object 
 9   host_response_rate              3295 non-null   float64
 10  host_acceptance_rate            3045 non-null   float64
 11  host_is_superhost               3818 non-null   bool   
 12  host_neighbourhood              38

In [ ]:
listings.shape

(3818, 64)

Checking for outliers

Col = price

In [ ]:
outliers_price = detect_outliers(listings, 'price')
print(f"Number of outliers in 'price': {len(outliers_price)}")
display(outliers_price[['price', 'property_type', 'room_type']].nlargest(10, columns='price'))

Number of outliers in 'price': 246


,price,property_type,room_type
3122,1000.0,House,Entire home/apt
1208,999.0,Apartment,Entire home/apt
2,975.0,House,Entire home/apt
3443,950.0,House,Entire home/apt
1279,899.0,Apartment,Entire home/apt
565,775.0,Boat,Entire home/apt
2518,750.0,House,Entire home/apt
2566,750.0,House,Entire home/apt
239,749.0,House,Entire home/apt
19,700.0,House,Entire home/apt


In [ ]:
listings['price'].describe()

,price
count,3818.000000
mean,127.976166
std,90.250022
min,20.000000
25%,75.000000
50%,100.000000
75%,150.000000
max,1000.000000


The price outliers detected are genuine prices of the properties which are possible in real world scenario, so not dropping or capping these values.

Col = accommodates

In [ ]:
outliers_accommodates = detect_outliers(listings, 'accommodates')
print(f"Number of outliers in 'accommodates': {len(outliers_accommodates)}")
display(outliers_accommodates[['accommodates', 'property_type', 'room_type']].nlargest(10, columns='accommodates'))

Number of outliers in 'accommodates': 184


,accommodates,property_type,room_type
14,16,House,Entire home/apt
2868,16,House,Entire home/apt
3351,16,House,Entire home/apt
3803,16,House,Entire home/apt
601,15,House,Entire home/apt
1200,15,House,Entire home/apt
2091,14,House,Entire home/apt
2195,14,House,Entire home/apt
3577,14,House,Entire home/apt
67,12,House,Entire home/apt


In [ ]:
listings['accommodates'].describe()

,accommodates
count,3818.000000
mean,3.349398
std,1.977599
min,1.000000
25%,2.000000
50%,3.000000
75%,4.000000
max,16.000000


These are extreme outliers although the property types are mostly entire house, still 16 people are extreme values we can cap them to 99th percentile

In [ ]:
accommodates_99th_percentile = listings['accommodates'].quantile(0.99)
listings['accommodates'] = listings['accommodates'].clip(upper=accommodates_99th_percentile)

print(f"'accommodates' values capped at the 99th percentile: {accommodates_99th_percentile}")
print("Descriptive statistics for 'accommodates' after capping:")
display(listings['accommodates'].describe())

'accommodates' values capped at the 99th percentile: 10.0
Descriptive statistics for 'accommodates' after capping:


,accommodates
count,3818.000000
mean,3.328706
std,1.885543
min,1.000000
25%,2.000000
50%,3.000000
75%,4.000000
max,10.000000


Col = guests_included

In [ ]:
outliers_guests_included = detect_outliers(listings, 'guests_included')
print(f"Number of outliers in 'guests_included': {len(outliers_guests_included)}")
display(outliers_guests_included[['guests_included', 'property_type', 'room_type']].nlargest(10, columns='guests_included'))

Number of outliers in 'guests_included': 382


,guests_included,property_type,room_type
693,15,Apartment,Private room
2868,13,House,Entire home/apt
3803,12,House,Entire home/apt
601,11,House,Entire home/apt
2,10,House,Entire home/apt
927,10,House,Entire home/apt
184,9,House,Entire home/apt
461,9,House,Entire home/apt
14,8,House,Entire home/apt
158,8,Townhouse,Entire home/apt


In [ ]:
listings['guests_included'].describe()

,guests_included
count,3818.000000
mean,1.672603
std,1.311040
min,0.000000
25%,1.000000
50%,1.000000
75%,2.000000
max,15.000000


First we will detect and drop the rows which have 0 values

In [ ]:
print(f"Number of rows before dropping 'guests_included' == 0: {len(listings)}")
listings = listings[listings['guests_included'] != 0]
print(f"Number of rows after dropping 'guests_included' == 0: {len(listings)}")

print("Descriptive statistics for 'guests_included' after dropping 0 values:")
display(listings['guests_included'].describe())

Number of rows before dropping 'guests_included' == 0: 3818
Number of rows after dropping 'guests_included' == 0: 3726
Descriptive statistics for 'guests_included' after dropping 0 values:


,guests_included
count,3726.000000
mean,1.713902
std,1.300183
min,1.000000
25%,1.000000
50%,1.000000
75%,2.000000
max,15.000000


No we will clip the outliers to 99th percentile

In [ ]:
guests_included_99th_percentile = listings['guests_included'].quantile(0.99)
listings['guests_included'] = listings['guests_included'].clip(upper=guests_included_99th_percentile)

print(f"'guests_included' values capped at the 99th percentile: {guests_included_99th_percentile}")
print("Descriptive statistics for 'guests_included' after capping:")
display(listings['guests_included'].describe())

'guests_included' values capped at the 99th percentile: 7.0
Descriptive statistics for 'guests_included' after capping:


,guests_included
count,3726.000000
mean,1.698604
std,1.215397
min,1.000000
25%,1.000000
50%,1.000000
75%,2.000000
max,7.000000


Col = minimum_nights

In [ ]:
outliers_minimum_nights = detect_outliers(listings, 'minimum_nights')
print(f"Number of outliers in 'minimum_nights': {len(outliers_minimum_nights)}")
display(outliers_minimum_nights[['minimum_nights', 'property_type', 'room_type']].nlargest(10, columns='minimum_nights'))

# Adding the descriptive statistics for 'minimum_nights'
display(listings['minimum_nights'].describe())

Number of outliers in 'minimum_nights': 297


,minimum_nights,property_type,room_type
2617,1000,Apartment,Entire home/apt
2107,31,House,Entire home/apt
156,30,House,Private room
1144,30,Apartment,Private room
1746,30,Apartment,Private room
3114,30,Apartment,Entire home/apt
3462,30,House,Entire home/apt
3524,30,House,Entire home/apt
3753,29,House,Entire home/apt
2964,28,Apartment,Entire home/apt


,minimum_nights
count,3726.000000
mean,2.374665
std,16.501071
min,1.000000
25%,1.000000
50%,2.000000
75%,2.000000
max,1000.000000


These are extreme outliers we need to cap them at 99th percentile

In [ ]:
minimum_nights_99th_percentile = listings['minimum_nights'].quantile(0.99)
listings['minimum_nights'] = listings['minimum_nights'].clip(upper=minimum_nights_99th_percentile)

print(f"'minimum_nights' values capped at the 99th percentile: {minimum_nights_99th_percentile}")
print("Descriptive statistics for 'minimum_nights' after capping:")
display(listings['minimum_nights'].describe())

'minimum_nights' values capped at the 99th percentile: 13.75
Descriptive statistics for 'minimum_nights' after capping:


,minimum_nights
count,3726.000000
mean,2.047907
std,1.684021
min,1.000000
25%,1.000000
50%,2.000000
75%,2.000000
max,13.750000


In [ ]:
listings['minimum_nights'] = listings['minimum_nights'].astype(int)

print("Data type of 'minimum_nights' after conversion:")
display(listings['minimum_nights'].dtype)
print("Descriptive statistics for 'minimum_nights' after conversion:")
display(listings['minimum_nights'].describe())

Data type of 'minimum_nights' after conversion:


dtype('int64')

Descriptive statistics for 'minimum_nights' after conversion:


,minimum_nights
count,3726.000000
mean,2.040258
std,1.631729
min,1.000000
25%,1.000000
50%,2.000000
75%,2.000000
max,13.000000


Col = maximum_nights

In [ ]:
outliers_maximum_nights = detect_outliers(listings, 'maximum_nights')
print(f"Number of outliers in 'maximum_nights': {len(outliers_maximum_nights)}")
display(outliers_maximum_nights[['maximum_nights', 'property_type', 'room_type']].nlargest(10, columns='maximum_nights'))

Number of outliers in 'maximum_nights': 1


,maximum_nights,property_type,room_type
1432,100000,Apartment,Entire home/apt


In [ ]:
listings['maximum_nights'].describe()

,maximum_nights
count,3726.000000
mean,782.118626
std,1702.200549
min,1.000000
25%,60.000000
50%,1125.000000
75%,1125.000000
max,100000.000000


In [ ]:
maximum_nights_99th_percentile = listings['maximum_nights'].quantile(0.99)
listings['maximum_nights'] = listings['maximum_nights'].clip(upper=maximum_nights_99th_percentile)

print(f"'maximum_nights' values capped at the 99th percentile: {maximum_nights_99th_percentile}")
print("Descriptive statistics for 'maximum_nights' after capping:")
display(listings['maximum_nights'].describe())

'maximum_nights' values capped at the 99th percentile: 1125.0
Descriptive statistics for 'maximum_nights' after capping:


,maximum_nights
count,3726.000000
mean,755.394257
std,503.794051
min,1.000000
25%,60.000000
50%,1125.000000
75%,1125.000000
max,1125.000000


Col = number_of_reviews

In [ ]:
outliers_number_of_reviews = detect_outliers(listings, 'number_of_reviews')
print(f"Number of outliers in 'number_of_reviews': {len(outliers_number_of_reviews)}")
display(outliers_number_of_reviews[['number_of_reviews', 'property_type', 'room_type']].nlargest(10, columns='number_of_reviews'))

Number of outliers in 'number_of_reviews': 355


,number_of_reviews,property_type,room_type
1447,474,Loft,Private room
1441,466,Loft,Private room
3083,404,House,Entire home/apt
3600,353,House,Private room
2568,320,House,Entire home/apt
1499,319,Apartment,Entire home/apt
2166,297,Apartment,Entire home/apt
2690,294,Apartment,Private room
2456,287,House,Entire home/apt
2668,279,House,Private room


There are many features which wont add much value towards analysis, before importing the cleaned dataset only picking important features

In [ ]:
listings = listings[['name', 'host_name', 'host_since', 'host_response_time',
         'host_is_superhost', 'host_neighbourhood',  'host_listings_count', 'host_total_listings_count',
         'neighbourhood', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'latitude', 'longitude',
         'is_location_exact', 'property_type', 'room_type', 'accommodates', 'guests_included', 'bathrooms', 'bedrooms',
         'beds', 'bed_type', 'price', 'security_deposit', 'cleaning_fee',
         'extra_people', 'minimum_nights', 'maximum_nights', 'availability_30', 'availability_60',
         'availability_90', 'availability_365', 'number_of_reviews', 'review_scores_accuracy',
         'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_communication', 'review_scores_location',
         'review_scores_value', 'instant_bookable', 'cancellation_policy', 'has_reviews']]

These are genuine values

Here's a summary of all the data cleaning and preprocessing steps we've performed on the listings dataset:

Data Cleaning and Preprocessing Summary
Duplicate Removal: We checked for duplicate entries based on the id column and confirmed there were no duplicates in the dataset.

Feature Selection: We initially selected a subset of relevant columns from the original dataset for our analysis, reducing the DataFrame to 64 columns.

Data Type Conversion and Formatting:

Currency Columns: Columns like price, weekly_price, monthly_price, security_deposit, cleaning_fee, and extra_people were converted from string format (e.g., '$1,250.00') to `float64` by stripping '$$1,250.00') to `float64` by stripping '$' and ',' characters.
Percentage Columns: host_response_rate and host_acceptance_rate were converted from percentage strings (e.g., '95%') to float64 in a 0-1 range.
Boolean Columns: host_is_superhost, is_location_exact, has_availability, and instant_bookable were converted from 't'/'f' strings to boolean (True/False).
Handling Missing Values (Imputation and Dropping):

High Missingness: The square_feet column was dropped because it had more than 80% missing values.

Categorical Imputation: Object-type columns (e.g., transit, host_name, host_location, property_type) were imputed with 'unknown' for their missing values.

Mode Imputation: zipcode, market, and state were imputed with the mode of their respective neighbourhood_cleansed groups.

Median Imputation: bathrooms, bedrooms, and beds were imputed with the median values, conditioned on room_type and accommodates categories.

Zero Imputation: security_deposit and cleaning_fee nulls were imputed with 0.0, assuming no fee was charged.

Review Scores: For review_scores_* columns, an indicator flag has_reviews was created (1 if reviews exist, 0 if not), and then missing review scores were imputed with the column's median.

Outlier Detection and Handling:

Price: Outliers were detected but kept as they were considered genuine price variations.

Accommodates: Outliers in accommodates were capped at the 99th percentile.
Guests Included: Rows where guests_included was 0 were dropped, and then the remaining outliers were capped at the 99th percentile.

Minimum Nights: Outliers in minimum_nights were capped at the 99th percentile and the column was converted to an integer type.

Maximum Nights: Outliers in maximum_nights were capped at the 99th percentile.
Number of Reviews: Outliers were detected but not explicitly handled, as they were deemed genuine.

Reviews Per Month: Outliers were detected, but no explicit handling has been applied yet.

Importing the cleaned dataset for EDA and hypothesis testing

In [ ]:
listings.to_csv('airbnb_cleaned.csv', index=False)